# **Homework on Credit Risk**
## **Building an Application Scoring Model**

### **General Information**
- **Date assigned:** December 2, 2025  
- **Soft deadline:** 23:59 MSK, December 15, 2025  
- **Hard deadline:** 23:59 MSK, December 18, 2025  
- **Submission:** send your work to  
  \texttt{maria.vorobyova.ser@gmail.com}  
  with the subject format:
  \[
  \text{HSE\_CS\_[track]\_FullName}
  \]
  Example:
  \[
  \text{HSE\_CS\_PAD\_IVANOV\_IVAN\_IVANOVICH}
  \]

---

### **Grading and Penalties**
Maximum score: **10 points**

Late penalty:
\[
\text{Final Score} = 10 - \text{days late}
\]

Submission **after** the hard deadline is **not accepted**.

Work must be completed **independently**.  
Similar solutions → **plagiarism** → score **0**.

---

### **Score Reduction If**
- no comments in the notebook
- unclear or poorly written code
- incorrect analysis and conclusions

---

### **Task**
Build a scoring model estimating the **probability of default** at the **credit application stage**.

Follow the provided notebook strictly and complete every block.

---

### **Dataset**
Based on Kaggle competition:
\[
\text{Give Me Some Credit}
\]

Data source:  
https://www.kaggle.com/competitions/GiveMeSomeCredit/data  

Data description:  
**Data Dictionary.xlsx**



# **Work assignment:**
**1.Explatory Data Analysis - (Task weight: 20%)**

**2.Creating additional variables - (Task weight: 10%)**

**3. Model building (A logistic regression must be built on the WoE variables.)- (Task weight: 50%)**

**4. Using methods to reduce class imbalance - (Task weight: 20%)**

# **Submitting results:**

* Submit homework via the Yandex form as a link to your GitHub, where all files and code (Python) will be.
* GitHub must be open and the code must be working, without errors.
* Name the repository using the template (HW_4_2025-FirstName_LastName).
* Link to the Yandex form: https://forms.yandex.ru/u/68eece24505690c23425594c

We wish you good luck!✌

# Additional explanations for the task

In [10]:
from google.colab import drive
import json
import zipfile

import pandas as pd

In [11]:
drive.mount('/content/drive')
df_train=pd.read_csv('/content/drive/MyDrive/cs-training.csv',index_col='Unnamed: 0')
df_test = pd.read_csv('/content/drive/MyDrive/cs-test.csv',index_col='Unnamed: 0')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
df_train

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
149996,0,0.040674,74,0,0.225131,2100.0,4,0,1,0,0.0
149997,0,0.299745,44,0,0.716562,5584.0,4,0,1,0,2.0
149998,0,0.246044,58,0,3870.000000,NaN,18,0,1,0,0.0
149999,0,0.000000,30,0,0.000000,5716.0,4,0,0,0,0.0


In [13]:
df_test

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,NaN,0.885519,43,0,0.177513,5700.0,4,0,0,0,0.0
2,NaN,0.463295,57,0,0.527237,9141.0,15,0,4,0,2.0
3,NaN,0.043275,59,0,0.687648,5083.0,12,0,1,0,2.0
4,NaN,0.280308,38,1,0.925961,3200.0,7,0,2,0,0.0
5,NaN,1.000000,27,0,0.019917,3865.0,4,0,0,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
101499,NaN,0.282653,24,0,0.068522,1400.0,5,0,0,0,0.0
101500,NaN,0.922156,36,3,0.934217,7615.0,8,0,2,0,4.0
101501,NaN,0.081596,70,0,836.000000,NaN,3,0,0,0,NaN
101502,NaN,0.335457,56,0,3568.000000,NaN,8,0,2,1,3.0


# 1.Explatory Data Analysis. Максимально - (20%-2 балла)

- 0 points if the task is not completed
- 1 point if statistics are calculated and there are logical graphs (important, USEFUL graphs), but no conclusions are drawn
- 2 points if statistics are calculated and there are graphs (important, USEFUL graphs) and CONCLUSIONS are drawn (important, that the conclusions are correct)

In [14]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

df = df_train
print("Dataset shape:", df.shape)
print("Data types:\n", df.dtypes)
print("First rows:\n", df.head())
missing_df = pd.DataFrame({
    'MissingCount': df.isnull().sum(),
    'MissingShare': df.isnull().mean()
}).sort_values(by='MissingCount', ascending=False)

print("Missing values summary:\n", missing_df)

fig = px.bar(
    missing_df[missing_df.MissingCount > 0],
    y='MissingCount',
    title='Missing Values by Feature'
)
fig.show()
target_dist = df['SeriousDlqin2yrs'].value_counts(normalize=True)

print("Target distribution (share):\n", target_dist)

fig = px.bar(
    x=target_dist.index.astype(str),
    y=target_dist.values,
    labels={'x': 'Default (1 = Yes)', 'y': 'Proportion'},
    title='Target Variable Distribution'
)
fig.show()
grouped_stats = df.groupby('SeriousDlqin2yrs').median()
print("Median values by target:\n", grouped_stats)

viz_cols = [
    'RevolvingUtilizationOfUnsecuredLines',
    'DebtRatio',
    'MonthlyIncome'
]

df_viz = df.copy()
for col in viz_cols:
    upper = df[col].quantile(0.99)
    df_viz[col] = np.where(df[col] > upper, upper, df[col])

for col in viz_cols:
    fig = px.histogram(
        df_viz,
        x=col,
        nbins=50,
        title=f'Distribution of {col} (99th percentile)'
    )
    fig.show()

risk_vars = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse'
]

for col in risk_vars:
    fig = px.box(
        df,
        x='SeriousDlqin2yrs',
        y=col,
        title=f'{col} vs Default'
    )
    fig.show()

fig = px.box(df, x='SeriousDlqin2yrs', y='age', title='Age vs Default')
fig.show()

fig = px.box(
    df,
    x='SeriousDlqin2yrs',
    y='NumberOfOpenCreditLinesAndLoans',
    title='Open Credit Lines vs Default'
)
fig.show()
corr = df.corr()

fig = go.Figure(
    data=go.Heatmap(
        z=corr.values,
        x=corr.columns,
        y=corr.columns,
        colorscale='Viridis'
    )
)
fig.update_layout(title='Feature Correlation Heatmap')
fig.show()


Dataset shape: (150000, 11)

Data types:
 SeriousDlqin2yrs                          int64
RevolvingUtilizationOfUnsecuredLines    float64
age                                       int64
NumberOfTime30-59DaysPastDueNotWorse      int64
DebtRatio                               float64
MonthlyIncome                           float64
NumberOfOpenCreditLinesAndLoans           int64
NumberOfTimes90DaysLate                   int64
NumberRealEstateLoansOrLines              int64
NumberOfTime60-89DaysPastDueNotWorse      int64
NumberOfDependents                      float64
dtype: object

First rows:
    SeriousDlqin2yrs  RevolvingUtilizationOfUnsecuredLines  age  \
1                 1                              0.766127   45   
2                 0                              0.957151   40   
3                 0                              0.658180   38   
4                 0                              0.233810   30   
5                 0                              0.907239   49   

   Nu


Target distribution (share):
 SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64



Median values by target:
                   RevolvingUtilizationOfUnsecuredLines   age  \
SeriousDlqin2yrs                                               
0                                             0.133288  52.0   
1                                             0.838853  45.0   

                  NumberOfTime30-59DaysPastDueNotWorse  DebtRatio  \
SeriousDlqin2yrs                                                    
0                                                  0.0   0.362659   
1                                                  0.0   0.428227   

                  MonthlyIncome  NumberOfOpenCreditLinesAndLoans  \
SeriousDlqin2yrs                                                   
0                        5466.0                              8.0   
1                        4500.0                              7.0   

                  NumberOfTimes90DaysLate  NumberRealEstateLoansOrLines  \
SeriousDlqin2yrs                                                          
0               

## Exploratory Data Analysis (EDA)

### Dataset Overview
- Shape: **150,000 rows × 11 columns**  
- Target variable: `SeriousDlqin2yrs` (default within 2 years)
- Missing values:
  - `MonthlyIncome`: 19.8% missing
  - `NumberOfDependents`: 2.6% missing
  - Other features: no missing values

### Target Distribution
- Non-default: 93.3%  
- Default: 6.7%  
- Imbalanced dataset, which will affect modeling.

### Median Values by Target
- Defaulting borrowers have **higher credit utilization** (0.839 vs 0.133)  
- Slightly **lower income and fewer open credit lines**  
- Past-due counts and age differ moderately

### Visual Exploration
- Outlier capping applied for numeric variables (`RevolvingUtilizationOfUnsecuredLines`, `DebtRatio`, `MonthlyIncome`)  
- Histograms and boxplots show:
  - Skewed financial variables
  - Delinquency features clearly separate defaults from non-defaults
  - Age and credit lines show weaker separation

### Correlation
- Correlation heatmap indicates no extreme multicollinearity  
- Delinquency and utilization features most related to default


# 2.Creating additional variables - (Task weight: 10%)

Be creative: the more variables, the higher the score! However, variables must be logical; illogical variables will not be accepted.

- 0 points if the task is not completed.
- 0.5 points - 2 additional variables created.
- 1 point - more than 3 variables created.


In [15]:
def feature_engineering(df):
    df_fe = df.copy()

    df_fe['MonthlyIncome_missing'] = df_fe['MonthlyIncome'].isnull().astype(int)
    df_fe['NumberOfDependents_missing'] = df_fe['NumberOfDependents'].isnull().astype(int)

    df_fe['TotalPastDue'] = (
        df_fe['NumberOfTime30-59DaysPastDueNotWorse'] +
        df_fe['NumberOfTime60-89DaysPastDueNotWorse'] +
        df_fe['NumberOfTimes90DaysLate']
    )
    df_fe['AnyPastDueFlag'] = (df_fe['TotalPastDue'] > 0).astype(int)

    df_fe['HighUtilizationFlag'] = (df_fe['RevolvingUtilizationOfUnsecuredLines'] > 1).astype(int)
    df_fe['HighDebtRatioFlag'] = (df_fe['DebtRatio'] > 1).astype(int)

    df_fe['LogMonthlyIncome'] = np.log1p(df_fe['MonthlyIncome'])

    return df_fe

df_train_fe = feature_engineering(df_train)
df_test_fe = feature_engineering(df_test)

new_features = [
    'MonthlyIncome_missing',
    'NumberOfDependents_missing',
    'TotalPastDue',
    'AnyPastDueFlag',
    'HighUtilizationFlag',
    'HighDebtRatioFlag',
    'LogMonthlyIncome'
]

print("Additional variables created:", len(new_features))
print("New variables:")
for col in new_features:
    print("-", col)

print("Summary statistics of new variables (train):")
print(df_train_fe[new_features].describe())

binary_features = [
    'MonthlyIncome_missing',
    'NumberOfDependents_missing',
    'AnyPastDueFlag',
    'HighUtilizationFlag',
    'HighDebtRatioFlag'
]

print("Default rate by engineered binary features (train):")
for col in binary_features:
    print(f"\n{col}")
    print(df_train_fe.groupby(col)['SeriousDlqin2yrs'].agg(['count', 'mean']))

print("Correlation of engineered numeric features with target (train):")
for col in ['TotalPastDue', 'LogMonthlyIncome']:
    corr = df_train_fe[col].corr(df_train_fe['SeriousDlqin2yrs'])
    print(f"{col}: {corr:.4f}")


Number of additional variables created: 7

New variables:
- MonthlyIncome_missing
- NumberOfDependents_missing
- TotalPastDue
- AnyPastDueFlag
- HighUtilizationFlag
- HighDebtRatioFlag
- LogMonthlyIncome

Summary statistics of new variables (train):
       MonthlyIncome_missing  NumberOfDependents_missing   TotalPastDue  \
count          150000.000000               150000.000000  150000.000000   
mean                0.198207                    0.026160       0.927393   
std                 0.398650                    0.159611      12.466204   
min                 0.000000                    0.000000       0.000000   
25%                 0.000000                    0.000000       0.000000   
50%                 0.000000                    0.000000       0.000000   
75%                 0.000000                    0.000000       0.000000   
max                 1.000000                    1.000000     294.000000   

       AnyPastDueFlag  HighUtilizationFlag  HighDebtRatioFlag  \
count   1

## Feature Engineering

### Purpose
Additional variables were created to capture borrower behavior, financial stress, and data quality effects that are not fully reflected in the original dataset.

### Engineered Variables
- **Delinquency behavior:** `TotalPastDue`, `AnyPastDueFlag`
- **Financial stress indicators:** `HighUtilizationFlag`, `HighDebtRatioFlag`
- **Income and data quality:** `LogMonthlyIncome`, `MonthlyIncome_missing`, `NumberOfDependents_missing`

**Total number of additional variables:** 7

### Descriptive Statistics (train)
- Binary variables are sparse, indicating that they capture specific risk conditions rather than general population characteristics.
- `TotalPastDue` is highly right-skewed, reflecting rare but severe delinquency behavior.
- `LogMonthlyIncome` reduces skewness relative to raw income and provides a more stable numeric feature.

| Feature | Count | Mean | Std | Min | 25% | 50% | 75% | Max |
|---------|-------|------|-----|-----|-----|-----|-----|-----|
| MonthlyIncome_missing | 150000 | 0.198 | 0.399 | 0 | 0 | 0 | 0 | 1 |
| NumberOfDependents_missing | 150000 | 0.026 | 0.160 | 0 | 0 | 0 | 0 | 1 |
| TotalPastDue | 150000 | 0.927 | 12.466 | 0 | 0 | 0 | 0 | 294 |
| AnyPastDueFlag | 150000 | 0.202 | 0.402 | 0 | 0 | 0 | 0 | 1 |
| HighUtilizationFlag | 150000 | 0.022 | 0.147 | 0 | 0 | 0 | 0 | 1 |
| HighDebtRatioFlag | 150000 | 0.234 | 0.424 | 0 | 0 | 0 | 0 | 1 |
| LogMonthlyIncome | 120269 | 8.411 | 1.333 | 0 | 8.132 | 8.594 | 9.018 | 14.917 |

### Relationship with Default (train)
- **MonthlyIncome_missing:** missing income is associated with slightly lower default rate (5.6% vs 6.9%)
- **NumberOfDependents_missing:** missing dependents associated with lower default (4.6% vs 6.7%)
- **AnyPastDueFlag:** presence of any past due strongly increases default (22.3% vs 2.7%)
- **HighUtilizationFlag:** high credit utilization associated with higher default (37.2% vs 6.0%)
- **HighDebtRatioFlag:** minimal separation (6.5% vs 6.7%)

### Correlation with Target (train)
- `TotalPastDue`: 0.1155  
- `LogMonthlyIncome`: -0.0179  



# 3. Model building - (Task weight: 50%)
A logistic regression must be built on the WoE variables.

- If any other model is built, the score is 0.

A WoE transformation must be calculated - maximum 3 points.

The WoE calculation must be done in two steps:

Step 1. Fine Classification (1 point). Splitting into a large number of bins (intervals).
- Typically, interval variables are divided into 20, 30, and so on intervals.
- For categorical variables, one category per group.
- Next, calculate the WoE for each group.

Step 2. Coarse Classification (2 points). Consolidating the intervals obtained in Step 1. The result should be no more than 5-10 intervals.
- Typically, groups with similar WoE values ​​are combined.
- The WoE must be monotonic, meaning that after your combination, the result must be interpretable (it is necessary to graphically demonstrate that the WoE is monotonic).


In [17]:
X_train = df_train_fe.drop(columns=['SeriousDlqin2yrs'])
y_train = df_train_fe['SeriousDlqin2yrs']

X_test = df_test_fe.drop(columns=['SeriousDlqin2yrs'], errors='ignore')

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Target mean (train):", y_train.mean())

def fine_binning(x, y, bins=20):
    df_tmp = pd.DataFrame({'x': x, 'y': y})
    df_tmp['bin'] = pd.qcut(df_tmp['x'], q=bins, duplicates='drop')
    return df_tmp.groupby('bin')['y'].agg(['count', 'sum'])

def calculate_woe(df_bin, total_good, total_bad):
    df_bin['bad'] = df_bin['sum']
    df_bin['good'] = df_bin['count'] - df_bin['bad']
    df_bin['woe'] = np.log(
        (df_bin['good'] / total_good) /
        (df_bin['bad'] / total_bad)
    )
    return df_bin[['woe']]

total_good = (y_train == 0).sum()
total_bad = (y_train == 1).sum()

woe_variables = [
    'TotalPastDue',
    'RevolvingUtilizationOfUnsecuredLines',
    'DebtRatio',
    'age',
    'LogMonthlyIncome'
]

woe_maps = {}

for var in woe_variables:
    fine = fine_binning(X_train[var], y_train, bins=20)
    fine_woe = calculate_woe(fine, total_good, total_bad)
    fine_woe = fine_woe.reset_index()
    fine_woe['group'] = pd.qcut(fine_woe.index, q=5, duplicates='drop')
    coarse_woe = fine_woe.groupby('group')['woe'].mean()

    fig = px.line(
        x=range(len(coarse_woe)),
        y=coarse_woe.values,
        title=f'Monotonic WoE for {var}'
    )
    fig.show()

    bins = pd.qcut(X_train[var], q=5, duplicates='drop')
    woe_maps[var] = dict(zip(bins.cat.categories, coarse_woe.values))

def apply_woe(series, woe_map):
    binned = pd.cut(series, bins=list(woe_map.keys()))
    mapped = binned.map(woe_map)
    return pd.to_numeric(mapped, errors='coerce')

X_train_woe = pd.DataFrame()
X_test_woe = pd.DataFrame()

for var in woe_variables:
    X_train_woe[var + '_woe'] = apply_woe(X_train[var], woe_maps[var])
    X_test_woe[var + '_woe'] = apply_woe(X_test[var], woe_maps[var])

X_train_woe = X_train_woe.fillna(0)
X_test_woe = X_test_woe.fillna(0)

print("WoE-transformed train sample:")
print(X_train_woe.head())


Train shape: (150000, 17)
Test shape: (101503, 17)
Target mean (train): 0.06684


/tmp/ipython-input-1299364750.py:27: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipython-input-1299364750.py:62: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



/tmp/ipython-input-1299364750.py:27: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipython-input-1299364750.py:62: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



/tmp/ipython-input-1299364750.py:27: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipython-input-1299364750.py:62: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



/tmp/ipython-input-1299364750.py:27: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipython-input-1299364750.py:62: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



/tmp/ipython-input-1299364750.py:27: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipython-input-1299364750.py:62: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




WoE-transformed train sample:
   TotalPastDue_woe  RevolvingUtilizationOfUnsecuredLines_woe  DebtRatio_woe  \
1           0.00000                                 -1.331819      -0.411478   
2           0.56198                                 -1.331819       0.106636   
3           0.00000                                 -0.013551       0.106636   
4           0.56198                                  0.767946       0.106636   
5           0.56198                                 -1.331819       0.106636   

    age_woe  LogMonthlyIncome_woe  
1 -0.248960              0.420953  
2 -0.248960             -0.314969  
3 -0.487752             -0.274341  
4 -0.487752             -0.274341  
5 -0.065785              0.420953  


Constructing and evaluating logistic regression - maximum 1 point
- Constructing only logistic regression - 0.3 points
- Evaluating the model (roc auc, f1, etc.) - 0.3 points
- Constructing a scorecard - 0.4 points

As a reminder, the following formulas are required for the scorecard (details in the lecture and seminar):
Score_i =  (βi × WoE_i + α/n) × Factor + Offset/n, где

- Factor = pdo/ln(2)

- Offset = Target Score — (Factor × ln(Target Odds))

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_woe, y_train, test_size=0.2, random_state=42, stratify=y_train
)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_tr, y_tr)

y_tr_proba = lr.predict_proba(X_tr)[:, 1]
y_val_proba = lr.predict_proba(X_val)[:, 1]
y_val_pred = lr.predict(X_val)

print("ROC-AUC (train subset):", roc_auc_score(y_tr, y_tr_proba))
print("ROC-AUC (validation):", roc_auc_score(y_val, y_val_proba))
print("F1-score (validation):", f1_score(y_val, y_val_pred))

pdo = 20
target_score = 600
target_odds = 50

Factor = pdo / np.log(2)
Offset = target_score - Factor * np.log(target_odds)

scorecard = pd.DataFrame({
    'Feature': X_train_woe.columns,
    'Beta': lr.coef_[0]
})

scorecard['Score'] = scorecard['Beta'] * Factor
scorecard['BaseScore'] = Offset / len(scorecard)

print("Scorecard table:")
print(scorecard)

def calculate_psi(train, test, bins=10):
    psi = 0
    for col in train.columns:
        train_dist, bin_edges = np.histogram(train[col], bins=bins)
        test_dist, _ = np.histogram(test[col], bins=bin_edges)

        train_pct = train_dist / len(train)
        test_pct = test_dist / len(test)

        psi += np.sum(
            (train_pct - test_pct) *
            np.log((train_pct + 1e-6) / (test_pct + 1e-6))
        )
    return psi

psi_value = calculate_psi(X_train_woe, X_test_woe)
print("PSI (Train vs Test features):", psi_value)


ROC-AUC (train subset): 0.8345587625034674
ROC-AUC (validation): 0.8398692766209143
F1-score (validation): 0.20461783439490447

Scorecard table:
                                    Feature      Beta       Score  BaseScore
0                          TotalPastDue_woe -3.494405 -100.827218  97.424575
1  RevolvingUtilizationOfUnsecuredLines_woe -0.672967  -19.417735  97.424575
2                             DebtRatio_woe -0.590917  -17.050249  97.424575
3                                   age_woe -0.445650  -12.858741  97.424575
4                      LogMonthlyIncome_woe -0.349693  -10.090007  97.424575

PSI (Train vs Test features): 0.00024226430009863397


Conduct a sample stability analysis using PSI
- Compare the test and training samples you downloaded from Kaggle (df_train, df_test)

## Model Building: Logistic Regression on WoE Features

### Purpose
Built a **credit scoring model** using logistic regression on **WoE-transformed features**.  
WoE will help to capture the relationship between feature values and default probability while keeping the model interpretable.

### WoE Transformation
- Variables transformed: `TotalPastDue`, `RevolvingUtilizationOfUnsecuredLines`, `DebtRatio`, `age`, `LogMonthlyIncome`.
- Fine classification: split each variable into 20 bins.
- Coarse classification: merged bins into 5 groups with roughly monotonic WoE.
- WoE features are numeric and ready for logistic regression.

**Sample WoE-transformed train data:**

| TotalPastDue_woe | RevolvingUtilizationOfUnsecuredLines_woe | DebtRatio_woe | age_woe | LogMonthlyIncome_woe |
|-----------------|-----------------------------------------|---------------|---------|---------------------|
| 0.00000         | -1.331819                                | -0.411478     | -0.248960 | 0.420953            |
| 0.56198         | -1.331819                                | 0.106636      | -0.248960 | -0.314969           |
| 0.00000         | -0.013551                                | 0.106636      | -0.487752 | -0.274341           |

### Logistic Regression Performance
- **ROC-AUC (train subset):** 0.835  
- **ROC-AUC (validation):** 0.840  
- **F1-score (validation):** 0.205  

Interpretation:
- Model separates defaults and non-defaults well.
- F1-score is low due to rare default events (≈6.7%).

### Scorecard
| Feature                           | Beta      | Score      | BaseScore |
|----------------------------------|----------|-----------|-----------|
| TotalPastDue_woe                  | -3.494   | -100.83   | 97.42     |
| RevolvingUtilizationOfUnsecuredLines_woe | -0.673   | -19.42    | 97.42     |
| DebtRatio_woe                     | -0.591   | -17.05    | 97.42     |
| age_woe                           | -0.446   | -12.86    | 97.42     |
| LogMonthlyIncome_woe              | -0.350   | -10.09    | 97.42     |

- `TotalPastDue` has the largest effect on score.  
- Other features have moderate impact, all are intuitive.

### Population Stability Index (PSI)
- PSI (train vs test) = 0.00024  
- Very low PSI → no big changes in feature distribution between train and test.

### Conclusion
- Logistic regression on WoE features works well and is interpretable.  
- Delinquency features like `TotalPastDue` are most important.  
- Scorecard can convert model output to credit scores.  
- PSI shows model features are stable and reliable.


# 4. Using methods to reduce class imbalance - (Task weight: 20%)
- Try several methods to reduce class imbalance
- Choose the one that brings the greatest improvement

In [20]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_woe, y_train, test_size=0.2, random_state=42, stratify=y_train
)

results = {}

lr = LogisticRegression(max_iter=1000)
lr.fit(X_tr, y_tr)
y_val_proba = lr.predict_proba(X_val)[:, 1]
results['Original'] = roc_auc_score(y_val, y_val_proba)

ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_tr, y_tr)
lr.fit(X_ros, y_ros)
y_val_proba = lr.predict_proba(X_val)[:, 1]
results['RandomOversample'] = roc_auc_score(y_val, y_val_proba)

rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_tr, y_tr)
lr.fit(X_rus, y_rus)
y_val_proba = lr.predict_proba(X_val)[:, 1]
results['RandomUndersample'] = roc_auc_score(y_val, y_val_proba)

smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X_tr, y_tr)
lr.fit(X_sm, y_sm)
y_val_proba = lr.predict_proba(X_val)[:, 1]
results['SMOTE'] = roc_auc_score(y_val, y_val_proba)

results_df = pd.DataFrame.from_dict(results, orient='index', columns=['ROC-AUC'])
results_df = results_df.sort_values(by='ROC-AUC', ascending=False)
print("ROC-AUC comparison:")
print(results_df)


ROC-AUC comparison for imbalance methods:
                    ROC-AUC
Original           0.839869
RandomOversample   0.839755
SMOTE              0.839671
RandomUndersample  0.839546


- Tested methods: Original, Random Oversampling, SMOTE, Random Undersampling.  
- All methods show **very similar performance**.  
- ROC-AUC is not really improved, so originally it was enough.

_Optional, for those who have reached the end of the laptop_ 😊

What was your impression of the work?
What was difficult?
What was interesting?